In [ ]:
#| hide
import os
from pathlib import Path
from shutil import rmtree
from tempfile import gettempdir

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.nbio import write_nb as _write_nb

from nbskill.convert import py2nb
from nbskill.execute import exec_nb
from nbskill.mcp import create_mcp
from nbskill.read import chapter_context, file_context, project_context, symbol_context
from nbskill.review import diff_nb, style_check
from nbskill.write import update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists(): return folder
    return start


project_root = _find_project_root()
workspace = None


def _readme_workspace():
    global workspace
    default = Path(gettempdir()) / "nbskill-readme-demo"
    workspace = Path(os.environ.get("NBSKILL_README_DEMO_DIR", default))
    if workspace.exists(): rmtree(workspace)
    workspace.mkdir(parents=True)
    return workspace

# nbskill

`nbskill` is a small toolkit for working with nbdev notebooks as source code. It gives agents and humans notebook-aware commands for reading, writing, executing, reviewing, converting, and serving notebooks through MCP.

The central idea is simple: keep the notebook as the source of truth, but give automation stable handles so it can make small, reviewable changes without touching raw JSON.

## The problem this project solves

Raw notebooks are awkward for coding agents. Cell ids, outputs, metadata, Markdown, and code are mixed together in JSON, while nbdev expects the meaningful implementation to stay in notebooks and export clean Python modules from there.

That mismatch matters in practice. A small source edit can accidentally preserve stale outputs, overwrite the wrong cell after context goes stale, or bypass the notebook story that explains why the code exists. `nbskill` gives agents a narrow set of notebook-aware tools: context readers, structured edits, execution, diffs, and diagnostics.

## How the notebooks fit together

The notebooks in `nbs/` are ordered like the toolchain itself:

1. `00_foundation.ipynb` defines the private parsing, cell, chapter, and CLI helpers used everywhere else.
2. `01_read.ipynb` makes notebooks readable without JSON noise.
3. `02_write.ipynb` applies safe cell edits and exports when requested.
4. `03_execute.ipynb` executes notebooks in the local project context.
5. `04_review.ipynb` keeps review centered on code-cell diffs and style feedback.
6. `05_convert.ipynb` turns Python files into nbdev notebooks.
7. `06_skill.ipynb` installs the bundled agent skill.
8. `07_mcp.ipynb` wraps the functions as an MCP server.
9. `08_edit_interactive.ipynb` runs a bounded edit loop over one notebook.
10. `09_parallel.ipynb` provides locks so concurrent notebook operations stay orderly.
11. `10_graph.ipynb` builds a static symbol graph for definitions, callers, and private-helper reports.
12. `11_agent_workbench.ipynb` compiles taste, context, budgets, and gates into a small-diff agent workbench.

## Production readiness

Production readiness starts with contracts, then behavior, then cleanup. The repository treats notebooks as source, so every public feature needs a small executable contract before cells are split or code is moved.

The production core is intentionally narrow:

| Area | CLI | MCP | Public API | Status |
| --- | --- | --- | --- | --- |
| Reading | `project_context`, `file_context`, `chapter_context`, `symbol_context` | yes | yes | core |
| Writing | `write_nb`, `update_cell`, `batch_edit_nb` | yes | yes | core |
| Execution/review | `exec_nb`, `diff_nb`, `style_check`, `doctor` | yes | yes | core |
| Conversion | `py2nb`, `py2nbdev` | yes | yes | supporting |


### Production workflow

1. Edit the source notebook, not generated Python.
2. Keep the intended behavior covered by a contract cell or behavior-test notebook.
3. Export generated modules through the notebook-aware write path.
4. Run `uv run nbskill_validate nbs`.
5. Run focused execution checks with `uv run exec_nb <notebook> --check_only`.
6. Run `uv run style_check nbs --changed_only --max_diagnostics 80`.
7. Only promote strict style gates after the current warning backlog has been paid down.


## A tiny notebook to work on

The examples below create a temporary notebook, then use the same public functions that are exposed as CLI commands and MCP tools. Nothing here edits this repository.

In [ ]:
workspace = _readme_workspace()
demo_nb = workspace / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("## A tiny notebook", cell_type="markdown"),
    mk_cell("x = 2\nx + 3", cell_type="code"),
]), demo_nb)

## Reading: choose the smallest useful context

The context API has four deliberate levels. `project_context` orients you in a repository with README excerpts, notebook filenames, and notebook docstrings. `file_context` shows one notebook's imports, header docs, Markdown cells, and definition summaries, with `include_re` and `exclude_re` filters. `chapter_context` preserves the chapter-focused view for one selected section. `symbol_context` replaces symbol docs with exact implementation context, nearby prose, examples/tests, callers, and depth-controlled callees.

For CLI/Python use, `verbose=False` returns the structured payload without printing it; MCP tools keep that print control hidden.

In [ ]:
file_context(str(demo_nb))

Cell id=2054f7fb: markdown
## A tiny notebook


## Writing: add cells without raw notebook JSON

`write_nb` accepts cell blocks separated by `---`. The `%%markdown` and `%%code` markers make each new cell explicit, while `` keeps this temporary example from running nbdev export.

This is useful when adding examples, tests, or explanatory sections. The caller describes notebook cells as cells, not JSON objects, so nbskill can preserve notebook structure and clear stale execution state where needed.

For larger notebook refactors, the CLI-only `split_nb_chapter` command moves one `##` chapter into a new nbdev notebook. It creates the new `#| default_exp`, copies imports used by the moved code, imports source-notebook definitions still needed by the split-out chapter, and promotes referenced private helpers in the source notebook when needed. Run it as a dry run first; it is intentionally not exposed as an MCP tool. Pass `--no-dry_run` when the plan looks right.

In [ ]:
write_nb(str(demo_nb), chr(10).join([
    "%%markdown",
    "## Result",
    "The next cell computes from the earlier value.",
    "---",
    "%%code",
    "answer = x * 10",
    "answer",
]))
chapter_context(str(demo_nb), name="Result")

Wrote 4 cells to /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/nbskill-readme-demo/demo.ipynb
Cell id=4972a187: markdown
## Result
The next cell computes from the earlier value.

Cell id=fcf83b53: code
answer = x * 10
answer


## Updating: use ids for precise edits

A notebook cell id tells `update_cell` which cell to change. Use `old_str` or `line_range` when only part of the cell should change, or pass exactly one replacement cell block for a whole-cell update.

Use `split_before="def next_function"` to split an existing cell before a matching line, or `split=True` with `---` cell blocks to replace one large cell with several smaller cells.

Writes export automatically when the notebook has an nbdev export target, so generated Python stays in sync with the notebook source.


In [ ]:
answer_cell = next(cell for cell in _read_raw_nb(demo_nb).cells if "answer = x * 10" in cell.source)
update_cell(str(demo_nb), "answer = x * 12\nanswer", cell_id=answer_cell.id)
_ = chapter_context(str(demo_nb), any_cell_id=answer_cell.id)

Updated cell id=fcf83b53
Cell id=4972a187: markdown
1 | ## Result
2 | The next cell computes from the earlier value.

Cell id=fcf83b53: code
1 | answer = x * 12
2 | answer


## Executing: run the notebook as a notebook

`exec_nb` uses `execnb` and adds the notebook directory plus the project root to the import path. That lets tests and examples behave like they do inside an nbdev project.

This matters because many notebook bugs only appear when cells are run in order with the same imports, fixtures, and local package path a real user gets. A normal Python import check can miss that story; executing the notebook checks the literate source itself.

In [ ]:
_ = exec_nb(str(demo_nb), timeout=5, show_output=True)

## Reviewing: look at behavior and code changes

`symbol_context` answers the question "what should I know before changing this implementation?" with exact source, nearby Markdown, examples/tests, callers, and optional callee summaries. `diff_nb` keeps review focused on code-cell source rather than outputs and metadata.

These tools keep review at the level a maintainer cares about. `symbol_context` reconstructs the local rationale and impact around a function, while `diff_nb` filters out notebook churn so a reviewer can see whether the implementation changed.

In [ ]:
_ = symbol_context(str(project_root / "nbs/02_write.ipynb"), "write_nb", depth=0)
_ = diff_nb(str(project_root / "nbs/02_write.ipynb"), ref_a=None)

## Converting: bootstrap nbdev notebooks from Python

`py2nb` parses Python with `ast`, creates one nbdev notebook, and keeps exports explicit. It is useful when a project starts in `.py` files but wants to move toward literate notebooks.

In [ ]:
sample_py = workspace / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted_nb = workspace / "sample_tool.ipynb"

_ = py2nb(str(sample_py), dest=str(converted_nb))
_ = file_context(str(converted_nb))

Wrote 2 cells to /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/nbskill-readme-demo/sample_tool.ipynb
Cell id=py2nb-454dad7d: code
def double(x):


## Serving the workflow through MCP

The MCP server in `07_mcp.ipynb` registers the same operations as tools. The server layer is intentionally thin: it captures stdout, holds notebook locks around file operations, and delegates the real behavior back to the notebook-defined functions.

In [ ]:
mcp = create_mcp()
type(mcp).__name__

'FastMCP'

## A realistic agent workflow

A typical agent session should be small and reversible. First check that the MCP server is connected, then use the smallest reader that answers the current question.

```python
healthcheck()
project_context(path=".")
file_context(path="nbs/02_write.ipynb", include_re="write")
chapter_context(path="nbs/02_write.ipynb", name="Updating")
symbol_context(path="nbs/02_write.ipynb", symbol="write_nb", depth=1)
```

Then edit the cell that actually needs to change and run focused verification.

After inspecting the precise cell, edit by stable cell id or a narrow line range. Notebook writes export automatically when the notebook has an export target.

```python
update_cell(
    path="nbs/02_write.ipynb",
    cell_id="abc123",
    new="def target():
    return 'updated'",
)
```


Finish with a review or execution tool depending on what changed. Use `diff_nb` for implementation edits and `exec_nb` when the notebook behavior needs to be checked end to end.

```python
diff_nb(path="nbs/02_write.ipynb")
exec_nb(path="nbs/02_write.ipynb", timeout=10, show_output=True)
```


<!-- nbskill-skill:start -->
# Jupyter Notebooks

Use this skill when a repository treats notebooks as source files, especially nbdev projects where `nbs/*.ipynb` exports to Python modules. Prefer the nbskill MCP server as the normal interface: inspect, edit, execute, review, and diagnose notebooks through MCP tools instead of raw `.ipynb` JSON, generated `.py` files, or ad hoc shell commands.

## Setup

Install the local package and register the MCP server:

```bash
uv tool install --editable . --force
codex mcp add nbskill -- nbskill_mcp
claude mcp add nbskill -- nbskill_mcp
```

Call `healthcheck` first when MCP is available. If tools are missing, run `uv run nbskill_status`, reconnect the MCP server, and then return to the MCP workflow.

## Core Workflow

1. Use `healthcheck` before notebook work or after reinstalling/exporting tool signatures.
2. Use `project_context`, `file_context`, `chapter_context`, and `symbol_context` as context needs become more precise.
3. Use `symbol_graph` when you specifically need graph-oriented caller/callee impact beyond the symbol context payload.
4. Keep notebook craft in the loop: preserve the story, add rationale before code, and put examples or tests after implementation cells.
5. Verify with `exec_nb`, `diff_nb`, `style_check`, or `doctor` before handing work back.

## Notebook Craft

A good notebook has a story. Move one step at a time: describe the problem, show the small behavior, export the implementation, demonstrate it with visible output when useful, then protect it with a small test. Larger features should grow from earlier cells rather than appear as one big code block.

Keep cells small and semantic. A cell should usually be one of these things: Markdown rationale, imports, exported code, private implementation, a visible example, or a focused test. Avoid cells that mix several jobs, duplicate imports, hide unused code, or bundle many assertions together.

Documentation should explain the shape of the code, not just repeat it. Say why the behavior exists, what problem it solves, what tradeoff it chooses, and why an obvious alternative is not being used. For shared helpers, include cross-references: where the symbol is called, why those callers need it, and whether a private helper should be promoted before another notebook imports it.

Examples should be executable and useful to a reader. Prefer short examples close to the feature they demonstrate, with visible outputs when the output helps understanding. Tests should be small, local, and named by the single behavior they protect.

## CLI Fallback

Use CLI commands only when MCP tools are unavailable or final verification must run inside the project environment. Reconnect MCP as soon as practical; the CLI is a fallback, not the normal agent interface.

```bash
uv run nbskill_status
uv run project_context .
uv run file_context nbs/02_write.ipynb --include_re write
uv run chapter_context nbs/02_write.ipynb --name Updating
uv run symbol_context nbs/02_write.ipynb write_nb --depth 1
uv run batch_edit_nb --plan_file /tmp/plan.json --dry_run
uv run exec_nb nbs/02_write.ipynb --check_only
uv run style_check nbs --changed_only --max_diagnostics 80
```

Use `--help` on any CLI wrapper when shell quoting or parameter order is unclear.

## References

Open references only when the core workflow is not enough:

- `references/mcp-tools.md` for detailed MCP behavior, reconnect notes, and concurrency behavior.
- `references/mcp-tool-report.md` for MCP feature groups and tool-count reduction candidates.
- `references/cli-fallbacks.md` for shell-friendly command patterns.
- `references/conversion.md` for converting Python files or folders with `py2nb`.
- `references/extended-tools.md` for symbol docs, review, graph reports, and edit-interactive plans.
<!-- nbskill-skill:end -->

## Agent editing policy

Notebook edits should stay small, readable, and reviewable. Use MCP tools for normal notebook work, keep generated files in sync by exporting from the notebook source, and avoid raw notebook JSON unless you are diagnosing the tool itself.

A good edit improves both behavior and the surrounding explanation. Add or preserve rationale, cross-references, visible examples, and small tests when they help the notebook read as a coherent build-up rather than a pile of cells.

Do not hide broad changes inside oversized cells, duplicate imports across the same notebook scope, leave unused code behind, or bundle unrelated assertions into one test cell. Before stopping, inspect `diff_nb` and run a style-oriented check with `doctor(scopes="error,warning,style")` or `style_check` for the touched notebooks.